# ETF Constituents and Cap-Weighted Fundamentals

This notebook uses the EODHD API to:
1. Retrieve constituents (holdings) of ETFs
2. Get individual fundamental data for companies in each ETF
3. Calculate aggregated cap-weighted fundamentals for each ETF
4. Build a timeseries going back as far as possible, ending 2025-10-24

## Setup and Imports

In [16]:
import pandas as pd
import numpy as np
from urllib.request import urlopen
import certifi
import json
import os
import ssl
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import time
import warnings
warnings.filterwarnings('ignore')

# Environment variables
import dotenv
dotenv.load_dotenv()
EODHD_API_KEY = os.getenv('EODHD_API_KEY')

# Verify API key
print(f"EODHD API Key loaded: {bool(EODHD_API_KEY)}")
if EODHD_API_KEY:
    print(f"API Key (first 10 chars): {EODHD_API_KEY[:10]}...")
else:
    raise ValueError("EODHD_API_KEY not found in .env file!")

EODHD API Key loaded: True
API Key (first 10 chars): 6911297628...


## Load ETF Symbols

Load the ETF symbols from existing data files.

In [17]:
# Load the existing US equity data to get ETF symbols
all_etf_data = pd.read_csv('data/processed/all_etf_data.csv', index_col=0, header=[0, 1], parse_dates=True)

# Extract unique symbols (level 0 of MultiIndex columns)
etf_symbols = all_etf_data.columns.get_level_values(0).unique().tolist()

# Filter out index symbols (those starting with ^)
etf_symbols_only = [symbol for symbol in etf_symbols if not symbol.startswith('^')]

print(f"Total symbols found: {len(etf_symbols)}")
print(f"ETF symbols (excluding indices): {len(etf_symbols_only)}")
print(f"\nETF Symbols: {sorted(etf_symbols_only)}")

Total symbols found: 47
ETF symbols (excluding indices): 40

ETF Symbols: ['BIL', 'DIA', 'EWJ', 'FXI', 'IEF', 'INDA', 'IVE', 'IVW', 'IWB', 'IWC', 'IWD', 'IWF', 'IWM', 'IWN', 'IWO', 'IWR', 'IWV', 'ONEQ', 'QQQ', 'RSP', 'SHY', 'SPY', 'VEA', 'VGK', 'VOO', 'VPL', 'VTHR', 'VWO', 'VXUS', 'XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']


## API Helper Functions

In [18]:
def fetch_json_data(url, max_retries=3, delay=1):
    """
    Fetch JSON data from URL with retry logic and error handling
    
    Parameters:
    -----------
    url : str
        API endpoint URL
    max_retries : int
        Maximum number of retry attempts
    delay : float
        Delay between retries in seconds
    
    Returns:
    --------
    dict or None : Parsed JSON data or None if failed
    """
    for attempt in range(max_retries):
        try:
            context = ssl.create_default_context(cafile=certifi.where())
            response = urlopen(url, context=context, timeout=30)
            data = response.read().decode('utf-8')
            return json.loads(data)
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(delay)
                continue
            else:
                print(f"Error fetching data: {str(e)}")
                return None
    return None

## ETF Constituents Functions

In [19]:
def fetch_etf_constituents(etf_symbol, api_key=EODHD_API_KEY):
    """
    Fetch current constituents (holdings) of an ETF from EODHD API
    
    Parameters:
    -----------
    etf_symbol : str
        ETF ticker symbol (e.g., 'SPY')
    api_key : str
        EODHD API key
    
    Returns:
    --------
    pd.DataFrame : Constituents with columns [code, name, weight, exchange]
    """
    try:
        # EODHD fundamentals endpoint returns holdings data
        url = f'https://eodhd.com/api/fundamentals/{etf_symbol}.US?api_token={api_key}'
        
        data = fetch_json_data(url)
        
        if not data:
            print(f'  ✗ {etf_symbol}: No data returned')
            return None
        
        # Extract holdings from ETF_Data section
        if 'ETF_Data' in data and 'Holdings' in data['ETF_Data']:
            holdings = data['ETF_Data']['Holdings']
            
            if holdings and isinstance(holdings, dict):
                # Convert holdings dict to DataFrame
                holdings_list = []
                for ticker, info in holdings.items():
                    if isinstance(info, dict):
                        holdings_list.append({
                            'code': ticker,
                            'name': info.get('name', ''),
                            'weight': info.get('weight', 0),
                            'exchange': info.get('exchange', 'US')
                        })
                
                if holdings_list:
                    df = pd.DataFrame(holdings_list)
                    df['etf'] = etf_symbol
                    df['weight'] = pd.to_numeric(df['weight'], errors='coerce')
                    df = df.sort_values('weight', ascending=False).reset_index(drop=True)
                    
                    print(f'  ✓ {etf_symbol}: {len(df)} holdings (total weight: {df["weight"].sum():.2%})')
                    return df
        
        print(f'  ✗ {etf_symbol}: No holdings data found')
        return None
        
    except Exception as e:
        print(f'  ✗ {etf_symbol}: Error - {str(e)}')
        return None

## Company Fundamentals Functions

In [20]:
def fetch_company_fundamentals(ticker, exchange='US', api_key=EODHD_API_KEY):
    """
    Fetch fundamental data for a company from EODHD API
    
    Parameters:
    -----------
    ticker : str
        Stock ticker symbol (e.g., 'AAPL')
    exchange : str
        Exchange code (default: 'US')
    api_key : str
        EODHD API key
    
    Returns:
    --------
    dict : Fundamental data including financials, valuation, etc.
    """
    try:
        # Handle tickers that might already include exchange suffix
        if '.' in ticker:
            symbol_part = ticker
        else:
            symbol_part = f'{ticker}.{exchange}'
        
        url = f'https://eodhd.com/api/fundamentals/{symbol_part}?api_token={api_key}'
        data = fetch_json_data(url)
        
        if data and 'General' in data:
            return data
        return None
        
    except Exception as e:
        return None


def extract_fundamental_metrics(fundamentals, as_of_date=None):
    """
    Extract key fundamental metrics from company fundamentals data
    
    Parameters:
    -----------
    fundamentals : dict
        Full fundamentals data from EODHD
    as_of_date : str or None
        Date for which to extract metrics (YYYY-MM-DD)
    
    Returns:
    --------
    dict : Key metrics including P/E, P/B, ROE, debt ratios, etc.
    """
    if not fundamentals:
        return None
    
    metrics = {}
    
    # General information
    if 'General' in fundamentals:
        general = fundamentals['General']
        metrics['ticker'] = general.get('Code', '')
        metrics['name'] = general.get('Name', '')
        metrics['sector'] = general.get('Sector', '')
        metrics['industry'] = general.get('Industry', '')
    
    # Highlights - valuation and key metrics
    if 'Highlights' in fundamentals:
        highlights = fundamentals['Highlights']
        metrics['market_cap'] = highlights.get('MarketCapitalization', None)
        metrics['pe_ratio'] = highlights.get('PERatio', None)
        metrics['peg_ratio'] = highlights.get('PEGRatio', None)
        metrics['price_to_book'] = highlights.get('PriceToBookMRQ', None)
        metrics['price_to_sales'] = highlights.get('PriceToSalesTTM', None)
        metrics['dividend_yield'] = highlights.get('DividendYield', None)
        metrics['eps'] = highlights.get('EarningsShare', None)
        metrics['book_value_per_share'] = highlights.get('BookValueShareTTM', None)
        metrics['revenue_per_share'] = highlights.get('RevenuePerShareTTM', None)
        metrics['profit_margin'] = highlights.get('ProfitMargin', None)
        metrics['operating_margin'] = highlights.get('OperatingMarginTTM', None)
        metrics['roe'] = highlights.get('ReturnOnEquityTTM', None)
        metrics['roa'] = highlights.get('ReturnOnAssetsTTM', None)
        metrics['ebitda'] = highlights.get('EBITDA', None)
    
    # Valuation
    if 'Valuation' in fundamentals:
        valuation = fundamentals['Valuation']
        metrics['trailing_pe'] = valuation.get('TrailingPE', None)
        metrics['forward_pe'] = valuation.get('ForwardPE', None)
        metrics['price_to_sales_ttm'] = valuation.get('PriceSalesTTM', None)
        metrics['price_to_book_mrq'] = valuation.get('PriceBookMRQ', None)
        metrics['ev_to_revenue'] = valuation.get('EnterpriseValueRevenue', None)
        metrics['ev_to_ebitda'] = valuation.get('EnterpriseValueEbitda', None)
    
    # Technicals
    if 'Technicals' in fundamentals:
        technicals = fundamentals['Technicals']
        metrics['beta'] = technicals.get('Beta', None)
        metrics['52w_high'] = technicals.get('52WeekHigh', None)
        metrics['52w_low'] = technicals.get('52WeekLow', None)
    
    # SharesStats
    if 'SharesStats' in fundamentals:
        shares = fundamentals['SharesStats']
        metrics['shares_outstanding'] = shares.get('SharesOutstanding', None)
        metrics['shares_float'] = shares.get('SharesFloat', None)
    
    # Analyst Ratings
    if 'AnalystRatings' in fundamentals:
        ratings = fundamentals['AnalystRatings']
        metrics['analyst_rating'] = ratings.get('Rating', None)
        metrics['target_price'] = ratings.get('TargetPrice', None)
    
    return metrics


def get_quarterly_fundamentals_timeseries(ticker, start_date='2015-01-01', end_date='2025-10-24', exchange='US', api_key=EODHD_API_KEY):
    """
    Fetch historical quarterly fundamental data for a company
    
    Parameters:
    -----------
    ticker : str
        Stock ticker symbol
    start_date : str
        Start date (YYYY-MM-DD)
    end_date : str
        End date (YYYY-MM-DD)
    exchange : str
        Exchange code (default: 'US')
    api_key : str
        EODHD API key
    
    Returns:
    --------
    pd.DataFrame : Historical fundamental metrics with date index
    """
    try:
        # Handle tickers that might already include exchange suffix
        if '.' in ticker:
            symbol_part = ticker
        else:
            symbol_part = f'{ticker}.{exchange}'
        
        # Fetch current fundamentals which include historical financials
        url = f'https://eodhd.com/api/fundamentals/{symbol_part}?api_token={api_key}'
        data = fetch_json_data(url)
        
        if not data:
            return None
        
        # Extract quarterly financials
        quarterly_data = []
        
        # Income Statement quarterly
        if 'Financials' in data and 'Income_Statement' in data['Financials']:
            income_stmt = data['Financials']['Income_Statement']
            if 'quarterly' in income_stmt:
                for date_str, metrics in income_stmt['quarterly'].items():
                    date = pd.to_datetime(date_str)
                    if start_date <= date_str <= end_date:
                        quarterly_data.append({
                            'date': date,
                            'revenue': metrics.get('totalRevenue', None),
                            'gross_profit': metrics.get('grossProfit', None),
                            'operating_income': metrics.get('operatingIncome', None),
                            'net_income': metrics.get('netIncome', None),
                            'ebitda': metrics.get('ebitda', None),
                            'eps': metrics.get('eps', None),
                        })
        
        # Balance Sheet quarterly
        if 'Financials' in data and 'Balance_Sheet' in data['Financials']:
            balance_sheet = data['Financials']['Balance_Sheet']
            if 'quarterly' in balance_sheet:
                for i, (date_str, metrics) in enumerate(balance_sheet['quarterly'].items()):
                    date = pd.to_datetime(date_str)
                    if start_date <= date_str <= end_date:
                        # Find or create corresponding entry
                        matching = [d for d in quarterly_data if d['date'] == date]
                        if matching:
                            entry = matching[0]
                        else:
                            entry = {'date': date}
                            quarterly_data.append(entry)
                        
                        entry['total_assets'] = metrics.get('totalAssets', None)
                        entry['total_liabilities'] = metrics.get('totalLiab', None)
                        entry['total_equity'] = metrics.get('totalStockholderEquity', None)
                        entry['total_debt'] = metrics.get('totalDebt', None)
                        entry['current_assets'] = metrics.get('totalCurrentAssets', None)
                        entry['current_liabilities'] = metrics.get('totalCurrentLiabilities', None)
                        entry['cash'] = metrics.get('cash', None)
        
        # Cash Flow quarterly
        if 'Financials' in data and 'Cash_Flow' in data['Financials']:
            cash_flow = data['Financials']['Cash_Flow']
            if 'quarterly' in cash_flow:
                for date_str, metrics in cash_flow['quarterly'].items():
                    date = pd.to_datetime(date_str)
                    if start_date <= date_str <= end_date:
                        matching = [d for d in quarterly_data if d['date'] == date]
                        if matching:
                            entry = matching[0]
                        else:
                            entry = {'date': date}
                            quarterly_data.append(entry)
                        
                        entry['operating_cash_flow'] = metrics.get('totalCashFromOperatingActivities', None)
                        entry['investing_cash_flow'] = metrics.get('totalCashflowsFromInvestingActivities', None)
                        entry['financing_cash_flow'] = metrics.get('totalCashFromFinancingActivities', None)
                        entry['free_cash_flow'] = metrics.get('freeCashFlow', None)
        
        if not quarterly_data:
            return None
        
        # Convert to DataFrame
        df = pd.DataFrame(quarterly_data)
        df = df.sort_values('date').set_index('date')
        df['ticker'] = ticker
        
        # Calculate derived metrics
        df['gross_margin'] = df['gross_profit'] / df['revenue']
        df['operating_margin'] = df['operating_income'] / df['revenue']
        df['net_margin'] = df['net_income'] / df['revenue']
        df['roe'] = df['net_income'] / df['total_equity']
        df['roa'] = df['net_income'] / df['total_assets']
        df['debt_to_equity'] = df['total_debt'] / df['total_equity']
        df['current_ratio'] = df['current_assets'] / df['current_liabilities']
        
        return df
        
    except Exception as e:
        return None

## Calculate Cap-Weighted Aggregated Fundamentals

In [21]:
def calculate_weighted_fundamentals(constituents_df, fundamentals_dict, weight_col='weight'):
    """
    Calculate cap-weighted aggregated fundamental metrics for an ETF
    
    Parameters:
    -----------
    constituents_df : pd.DataFrame
        DataFrame with columns ['code', 'weight', ...]
    fundamentals_dict : dict
        Dictionary mapping ticker -> fundamental metrics dict
    weight_col : str
        Column name for weights
    
    Returns:
    --------
    dict : Aggregated weighted metrics
    """
    if constituents_df is None or constituents_df.empty:
        return None
    
    # Metrics to aggregate (must be numeric)
    metrics_to_aggregate = [
        'pe_ratio', 'price_to_book', 'price_to_sales', 'dividend_yield',
        'profit_margin', 'operating_margin', 'roe', 'roa',
        'beta', 'debt_to_equity', 'current_ratio',
        'gross_margin', 'net_margin', 'ev_to_ebitda'
    ]
    
    weighted_metrics = {}
    total_weight = 0
    
    for metric in metrics_to_aggregate:
        weighted_sum = 0
        valid_weight = 0
        
        for _, row in constituents_df.iterrows():
            ticker = row['code']
            weight = row[weight_col]
            
            if pd.isna(weight) or weight <= 0:
                continue
            
            if ticker in fundamentals_dict:
                fund = fundamentals_dict[ticker]
                if fund and metric in fund:
                    value = fund[metric]
                    if value is not None and not pd.isna(value):
                        try:
                            weighted_sum += float(value) * weight
                            valid_weight += weight
                        except (ValueError, TypeError):
                            continue
        
        if valid_weight > 0:
            weighted_metrics[metric] = weighted_sum / valid_weight
        else:
            weighted_metrics[metric] = None
    
    # Calculate total market cap
    total_market_cap = 0
    for _, row in constituents_df.iterrows():
        ticker = row['code']
        if ticker in fundamentals_dict:
            fund = fundamentals_dict[ticker]
            if fund and 'market_cap' in fund:
                mcap = fund['market_cap']
                if mcap is not None and not pd.isna(mcap):
                    try:
                        total_market_cap += float(mcap)
                    except (ValueError, TypeError):
                        continue
    
    weighted_metrics['total_market_cap'] = total_market_cap if total_market_cap > 0 else None
    weighted_metrics['num_constituents'] = len(constituents_df)
    weighted_metrics['total_weight'] = constituents_df[weight_col].sum()
    
    return weighted_metrics


def calculate_weighted_fundamentals_timeseries(constituents_df, quarterly_data_dict, weight_col='weight'):
    """
    Calculate time series of cap-weighted aggregated fundamental metrics
    
    Parameters:
    -----------
    constituents_df : pd.DataFrame
        DataFrame with columns ['code', 'weight', ...]
    quarterly_data_dict : dict
        Dictionary mapping ticker -> quarterly DataFrame
    weight_col : str
        Column name for weights
    
    Returns:
    --------
    pd.DataFrame : Time series of weighted metrics
    """
    if constituents_df is None or constituents_df.empty:
        return None
    
    # Get all unique dates from all constituents
    all_dates = set()
    for ticker in constituents_df['code']:
        if ticker in quarterly_data_dict and quarterly_data_dict[ticker] is not None:
            all_dates.update(quarterly_data_dict[ticker].index)
    
    if not all_dates:
        return None
    
    all_dates = sorted(list(all_dates))
    
    # Metrics to aggregate
    metrics_to_aggregate = [
        'revenue', 'gross_profit', 'operating_income', 'net_income', 'ebitda',
        'total_assets', 'total_liabilities', 'total_equity', 'total_debt',
        'operating_cash_flow', 'free_cash_flow',
        'gross_margin', 'operating_margin', 'net_margin', 'roe', 'roa',
        'debt_to_equity', 'current_ratio'
    ]
    
    timeseries_data = []
    
    for date in all_dates:
        date_metrics = {'date': date}
        
        for metric in metrics_to_aggregate:
            weighted_sum = 0
            valid_weight = 0
            
            for _, row in constituents_df.iterrows():
                ticker = row['code']
                weight = row[weight_col]
                
                if pd.isna(weight) or weight <= 0:
                    continue
                
                if ticker in quarterly_data_dict and quarterly_data_dict[ticker] is not None:
                    df = quarterly_data_dict[ticker]
                    
                    # Find closest date (forward fill)
                    if date in df.index:
                        value = df.loc[date, metric] if metric in df.columns else None
                    else:
                        # Use most recent data before this date
                        prior_dates = df.index[df.index <= date]
                        if len(prior_dates) > 0:
                            latest_date = prior_dates[-1]
                            value = df.loc[latest_date, metric] if metric in df.columns else None
                        else:
                            value = None
                    
                    if value is not None and not pd.isna(value):
                        try:
                            weighted_sum += float(value) * weight
                            valid_weight += weight
                        except (ValueError, TypeError):
                            continue
            
            if valid_weight > 0:
                date_metrics[metric] = weighted_sum / valid_weight
            else:
                date_metrics[metric] = None
        
        timeseries_data.append(date_metrics)
    
    if not timeseries_data:
        return None
    
    df = pd.DataFrame(timeseries_data)
    df = df.set_index('date').sort_index()
    
    return df

## Main Processing Pipeline

In [48]:
# Configuration
START_DATE = '1990-01-01'  # Start date for historical data
END_DATE = '2025-10-24'    # End date

# Select a subset of ETFs to process (or use all)
# For testing, you might want to start with a few ETFs
etfs_to_process = etf_symbols_only[11:12]  # Start with first 5 ETFs for testing
# etfs_to_process = etf_symbols_only  # Uncomment to process all ETFs

print(f"Processing {len(etfs_to_process)} ETFs")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"ETFs: {etfs_to_process}")

Processing 1 ETFs
Date range: 1990-01-01 to 2025-10-24
ETFs: ['SPY']


### Step 1: Fetch ETF Constituents

In [49]:
etfs_to_process

['SPY']

In [50]:
print("\n" + "="*80)
print("STEP 1: Fetching ETF Constituents")
print("="*80)

all_constituents = {}

for i, etf in enumerate(etfs_to_process, 1):
    print(f"\n[{i}/{len(etfs_to_process)}] {etf}:")
    constituents = fetch_etf_constituents(etf)
    
    if constituents is not None and not constituents.empty:
        all_constituents[etf] = constituents
        print(f"      Top 5 holdings:")
        for idx, row in constituents.head(5).iterrows():
            print(f"        {row['code']:6s} - {row['weight']:6.2%} - {row['name'][:40]}")
    
    # Rate limiting - pause between requests
    time.sleep(0.5)

print(f"\n{'='*80}")
print(f"Successfully retrieved constituents for {len(all_constituents)} / {len(etfs_to_process)} ETFs")
print(f"{'='*80}")


STEP 1: Fetching ETF Constituents

[1/1] SPY:
  ✓ SPY: 100 holdings (total weight: 0.00%)
      Top 5 holdings:
        NVDA.US -  0.00% - 
        SCHW.US -  0.00% - 
        ETN.US -  0.00% - 
        BA.US  -  0.00% - 
        GEV.US -  0.00% - 
  ✓ SPY: 100 holdings (total weight: 0.00%)
      Top 5 holdings:
        NVDA.US -  0.00% - 
        SCHW.US -  0.00% - 
        ETN.US -  0.00% - 
        BA.US  -  0.00% - 
        GEV.US -  0.00% - 

Successfully retrieved constituents for 1 / 1 ETFs

Successfully retrieved constituents for 1 / 1 ETFs


In [51]:
all_constituents['SPY']

,code,name,weight,exchange,etf
0,NVDA.US,,0,US,SPY
1,SCHW.US,,0,US,SPY
2,ETN.US,,0,US,SPY
3,BA.US,,0,US,SPY
4,GEV.US,,0,US,SPY
...,...,...,...,...,...
95,CSCO.US,,0,US,SPY
96,IBM.US,,0,US,SPY
97,UNH.US,,0,US,SPY
98,CVX.US,,0,US,SPY


### Debug: Check Constituent Ticker Formats

Let's examine the ticker formats to understand the 404 errors.

In [52]:
# Check the format of tickers in constituents
print("Sample tickers from constituents:")
for etf, constituents in all_constituents.items():
    print(f"\n{etf} - Top 10 holdings:")
    print(constituents[['code', 'name', 'exchange']].head(10))
    break  # Just show first ETF for now

Sample tickers from constituents:

SPY - Top 10 holdings:
      code name exchange
0  NVDA.US            US
1  SCHW.US            US
2   ETN.US            US
3    BA.US            US
4   GEV.US            US
5   ACN.US            US
6  GILD.US            US
7  SPGI.US            US
8   BLK.US            US
9  BKNG.US            US


### Step 2: Get Unique Stocks and Fetch Their Fundamentals

In [53]:
print("\n" + "="*80)
print("STEP 2: Identifying Unique Stocks")
print("="*80)

# Get all unique stocks across all ETFs
unique_stocks = set()
for etf, constituents in all_constituents.items():
    unique_stocks.update(constituents['code'].tolist())

unique_stocks = sorted(list(unique_stocks))
print(f"Total unique stocks across all ETFs: {len(unique_stocks)}")
print(f"Sample stocks: {unique_stocks[:10]}")


STEP 2: Identifying Unique Stocks
Total unique stocks across all ETFs: 100
Sample stocks: ['AAPL.US', 'ABBV.US', 'ABT.US', 'ACN.US', 'ADBE.US', 'ADI.US', 'AMAT.US', 'AMD.US', 'AMGN.US', 'AMZN.US']


### Step 3: Fetch Current Fundamentals for All Stocks

In [54]:
print("\n" + "="*80)
print("STEP 3: Fetching Current Fundamental Data for Stocks")
print("="*80)

stock_fundamentals = {}
stock_metrics = {}

for i, ticker in enumerate(unique_stocks, 1):
    if i % 10 == 0:
        print(f"Progress: {i}/{len(unique_stocks)} ({i/len(unique_stocks)*100:.1f}%)")
    
    # Fetch fundamental data
    fund_data = fetch_company_fundamentals(ticker)
    
    if fund_data:
        stock_fundamentals[ticker] = fund_data
        metrics = extract_fundamental_metrics(fund_data)
        if metrics:
            stock_metrics[ticker] = metrics
    
    # Rate limiting
    time.sleep(0.3)

print(f"\n{'='*80}")
print(f"Successfully retrieved fundamentals for {len(stock_fundamentals)} / {len(unique_stocks)} stocks")
print(f"{'='*80}")


STEP 3: Fetching Current Fundamental Data for Stocks


KeyboardInterrupt: 

### Step 3 (Improved): Fetch Current Fundamentals with Better Error Handling

In [55]:
print("\n" + "="*80)
print("STEP 3 (IMPROVED): Fetching Current Fundamental Data for Stocks")
print("="*80)

stock_fundamentals_v2 = {}
stock_metrics_v2 = {}
failed_tickers = []
success_count = 0

# Build a mapping of ticker to exchange from constituents data
ticker_exchange_map = {}
for etf, constituents in all_constituents.items():
    for _, row in constituents.iterrows():
        ticker = row['code']
        exchange = row.get('exchange', 'US')
        if ticker not in ticker_exchange_map:
            ticker_exchange_map[ticker] = exchange

print(f"Total unique stocks to fetch: {len(unique_stocks)}")
print(f"Sample ticker->exchange mappings: {dict(list(ticker_exchange_map.items())[:5])}")

for i, ticker in enumerate(unique_stocks, 1):
    if i % 10 == 0 or i == 1:
        print(f"Progress: {i}/{len(unique_stocks)} ({i/len(unique_stocks)*100:.1f}%) - Success: {success_count}, Failed: {len(failed_tickers)}")
    
    # Get the exchange for this ticker
    exchange = ticker_exchange_map.get(ticker, 'US')
    
    # Fetch fundamental data
    fund_data = fetch_company_fundamentals(ticker, exchange=exchange)
    
    if fund_data:
        stock_fundamentals_v2[ticker] = fund_data
        metrics = extract_fundamental_metrics(fund_data)
        if metrics:
            stock_metrics_v2[ticker] = metrics
            success_count += 1
    else:
        failed_tickers.append((ticker, exchange))
    
    # Rate limiting
    time.sleep(0.3)

print(f"\n{'='*80}")
print(f"Successfully retrieved fundamentals for {len(stock_fundamentals_v2)} / {len(unique_stocks)} stocks")
print(f"Failed tickers: {len(failed_tickers)}")
if failed_tickers[:10]:
    print(f"\nFirst 10 failed tickers:")
    for ticker, exchange in failed_tickers[:10]:
        print(f"  - {ticker} (exchange: {exchange})")
print(f"{'='*80}")

# Update the main variables
stock_fundamentals = stock_fundamentals_v2
stock_metrics = stock_metrics_v2


STEP 3 (IMPROVED): Fetching Current Fundamental Data for Stocks
Total unique stocks to fetch: 100
Sample ticker->exchange mappings: {'NVDA.US': 'US', 'SCHW.US': 'US', 'ETN.US': 'US', 'BA.US': 'US', 'GEV.US': 'US'}
Progress: 1/100 (1.0%) - Success: 0, Failed: 0
Progress: 10/100 (10.0%) - Success: 9, Failed: 0
Progress: 10/100 (10.0%) - Success: 9, Failed: 0
Progress: 20/100 (20.0%) - Success: 19, Failed: 0
Progress: 20/100 (20.0%) - Success: 19, Failed: 0
Progress: 30/100 (30.0%) - Success: 29, Failed: 0
Progress: 30/100 (30.0%) - Success: 29, Failed: 0
Progress: 40/100 (40.0%) - Success: 39, Failed: 0
Progress: 40/100 (40.0%) - Success: 39, Failed: 0
Progress: 50/100 (50.0%) - Success: 49, Failed: 0
Progress: 50/100 (50.0%) - Success: 49, Failed: 0
Progress: 60/100 (60.0%) - Success: 59, Failed: 0
Progress: 60/100 (60.0%) - Success: 59, Failed: 0
Progress: 70/100 (70.0%) - Success: 69, Failed: 0
Progress: 70/100 (70.0%) - Success: 69, Failed: 0
Progress: 80/100 (80.0%) - Success: 79, 

### Step 4: Fetch Historical Quarterly Fundamentals for All Stocks

In [15]:
print("\n" + "="*80)
print("STEP 4: Fetching Historical Quarterly Fundamental Data")
print("="*80)

stock_quarterly_data = {}

for i, ticker in enumerate(unique_stocks, 1):
    if i % 10 == 0:
        print(f"Progress: {i}/{len(unique_stocks)} ({i/len(unique_stocks)*100:.1f}%)")
    
    quarterly_df = get_quarterly_fundamentals_timeseries(ticker, START_DATE, END_DATE)
    
    if quarterly_df is not None and not quarterly_df.empty:
        stock_quarterly_data[ticker] = quarterly_df
    
    # Rate limiting
    time.sleep(0.3)

print(f"\n{'='*80}")
print(f"Successfully retrieved quarterly data for {len(stock_quarterly_data)} / {len(unique_stocks)} stocks")
print(f"{'='*80}")


STEP 4: Fetching Historical Quarterly Fundamental Data
Error fetching data: HTTP Error 404: Not Found
Error fetching data: HTTP Error 404: Not Found
Error fetching data: HTTP Error 404: Not Found
Error fetching data: HTTP Error 404: Not Found


KeyboardInterrupt: 

### Step 4 (Improved): Fetch Historical Quarterly Fundamentals with Better Tracking

In [56]:
print("\n" + "="*80)
print("STEP 4 (IMPROVED): Fetching Historical Quarterly Fundamental Data")
print("="*80)

stock_quarterly_data_v2 = {}
quarterly_success = 0

for i, ticker in enumerate(unique_stocks, 1):
    if i % 10 == 0 or i == 1:
        print(f"Progress: {i}/{len(unique_stocks)} ({i/len(unique_stocks)*100:.1f}%) - Quarterly data retrieved: {quarterly_success}")
    
    # Get the exchange for this ticker
    exchange = ticker_exchange_map.get(ticker, 'US')
    
    quarterly_df = get_quarterly_fundamentals_timeseries(ticker, START_DATE, END_DATE, exchange=exchange)
    
    if quarterly_df is not None and not quarterly_df.empty:
        stock_quarterly_data_v2[ticker] = quarterly_df
        quarterly_success += 1
    
    # Rate limiting
    time.sleep(0.3)

print(f"\n{'='*80}")
print(f"Successfully retrieved quarterly data for {len(stock_quarterly_data_v2)} / {len(unique_stocks)} stocks")
print(f"{'='*80}")

# Update the main variable
stock_quarterly_data = stock_quarterly_data_v2


STEP 4 (IMPROVED): Fetching Historical Quarterly Fundamental Data
Progress: 1/100 (1.0%) - Quarterly data retrieved: 0
Progress: 10/100 (10.0%) - Quarterly data retrieved: 0
Progress: 10/100 (10.0%) - Quarterly data retrieved: 0
Progress: 20/100 (20.0%) - Quarterly data retrieved: 0
Progress: 20/100 (20.0%) - Quarterly data retrieved: 0
Progress: 30/100 (30.0%) - Quarterly data retrieved: 0
Progress: 30/100 (30.0%) - Quarterly data retrieved: 0
Progress: 40/100 (40.0%) - Quarterly data retrieved: 0
Progress: 40/100 (40.0%) - Quarterly data retrieved: 0
Progress: 50/100 (50.0%) - Quarterly data retrieved: 0
Progress: 50/100 (50.0%) - Quarterly data retrieved: 0
Progress: 60/100 (60.0%) - Quarterly data retrieved: 0
Progress: 60/100 (60.0%) - Quarterly data retrieved: 0
Progress: 70/100 (70.0%) - Quarterly data retrieved: 0
Progress: 70/100 (70.0%) - Quarterly data retrieved: 0
Progress: 80/100 (80.0%) - Quarterly data retrieved: 0
Progress: 80/100 (80.0%) - Quarterly data retrieved: 0


### Step 5: Calculate Cap-Weighted Fundamentals for Each ETF

In [ ]:
print("\n" + "="*80)
print("STEP 5: Calculating Cap-Weighted Aggregated Fundamentals")
print("="*80)

etf_weighted_metrics = {}
etf_weighted_timeseries = {}

for etf, constituents in all_constituents.items():
    print(f"\nProcessing {etf}...")
    
    # Current weighted metrics
    weighted = calculate_weighted_fundamentals(constituents, stock_metrics)
    if weighted:
        etf_weighted_metrics[etf] = weighted
        print(f"  ✓ Current metrics calculated")
        print(f"    - P/E Ratio: {weighted.get('pe_ratio', 'N/A')}")
        print(f"    - ROE: {weighted.get('roe', 'N/A')}")
        print(f"    - Dividend Yield: {weighted.get('dividend_yield', 'N/A')}")
    
    # Historical weighted timeseries
    timeseries = calculate_weighted_fundamentals_timeseries(constituents, stock_quarterly_data)
    if timeseries is not None and not timeseries.empty:
        etf_weighted_timeseries[etf] = timeseries
        print(f"  ✓ Time series calculated")
        print(f"    - Date range: {timeseries.index.min().date()} to {timeseries.index.max().date()}")
        print(f"    - Observations: {len(timeseries)}")

print(f"\n{'='*80}")
print(f"Calculated metrics for {len(etf_weighted_metrics)} ETFs")
print(f"Calculated time series for {len(etf_weighted_timeseries)} ETFs")
print(f"{'='*80}")

### Step 6: Save Results

In [ ]:
print("\n" + "="*80)
print("STEP 6: Saving Results")
print("="*80)

# Save constituents data
all_constituents_df = pd.concat(all_constituents.values(), ignore_index=True)
all_constituents_df.to_csv('data/processed/etf_constituents.csv', index=False)
print(f"✓ Saved ETF constituents to: data/processed/etf_constituents.csv")

# Save current weighted metrics
if etf_weighted_metrics:
    weighted_metrics_df = pd.DataFrame(etf_weighted_metrics).T
    weighted_metrics_df.index.name = 'etf'
    weighted_metrics_df.to_csv('data/processed/etf_weighted_fundamentals_current.csv')
    print(f"✓ Saved current weighted metrics to: data/processed/etf_weighted_fundamentals_current.csv")

# Save time series data for each ETF
if etf_weighted_timeseries:
    for etf, ts_df in etf_weighted_timeseries.items():
        filename = f'data/processed/etf_weighted_fundamentals_{etf}_timeseries.csv'
        ts_df.to_csv(filename)
    print(f"✓ Saved {len(etf_weighted_timeseries)} ETF time series files")

# Save combined time series
if etf_weighted_timeseries:
    # Create a combined DataFrame with MultiIndex columns
    combined_ts = {}
    for etf, ts_df in etf_weighted_timeseries.items():
        for col in ts_df.columns:
            combined_ts[(etf, col)] = ts_df[col]
    
    combined_df = pd.DataFrame(combined_ts)
    combined_df.columns = pd.MultiIndex.from_tuples(combined_df.columns, names=['etf', 'metric'])
    combined_df.to_csv('data/processed/etf_weighted_fundamentals_all_timeseries.csv')
    print(f"✓ Saved combined time series to: data/processed/etf_weighted_fundamentals_all_timeseries.csv")
    print(f"  Shape: {combined_df.shape}")
    print(f"  Date range: {combined_df.index.min()} to {combined_df.index.max()}")

# Save stock fundamentals as JSON for reference
with open('data/processed/stock_fundamentals_raw.json', 'w') as f:
    json.dump(stock_fundamentals, f, indent=2)
print(f"✓ Saved raw stock fundamentals to: data/processed/stock_fundamentals_raw.json")

print(f"\n{'='*80}")
print("DATA PROCESSING COMPLETE")
print(f"{'='*80}")

## Summary and Verification

In [ ]:
print("\n" + "="*80)
print("ETF CAP-WEIGHTED FUNDAMENTALS SUMMARY")
print("="*80)

print(f"\n1. ETF Constituents:")
print(f"   - Total ETFs processed: {len(all_constituents)}")
print(f"   - Total unique stocks: {len(unique_stocks)}")
print(f"   - Average holdings per ETF: {all_constituents_df.groupby('etf').size().mean():.1f}")

print(f"\n2. Stock Fundamental Data:")
print(f"   - Stocks with current fundamentals: {len(stock_fundamentals)}")
print(f"   - Stocks with quarterly data: {len(stock_quarterly_data)}")
print(f"   - Coverage: {len(stock_fundamentals)/len(unique_stocks)*100:.1f}%")

print(f"\n3. ETF Weighted Fundamentals:")
print(f"   - ETFs with current weighted metrics: {len(etf_weighted_metrics)}")
print(f"   - ETFs with time series: {len(etf_weighted_timeseries)}")

if etf_weighted_timeseries:
    print(f"\n4. Time Series Details:")
    for etf, ts_df in etf_weighted_timeseries.items():
        print(f"   {etf}:")
        print(f"     - Date range: {ts_df.index.min().date()} to {ts_df.index.max().date()}")
        print(f"     - Observations: {len(ts_df)}")
        print(f"     - Metrics: {len(ts_df.columns)}")

print(f"\n5. Sample Current Metrics:")
if etf_weighted_metrics:
    sample_etf = list(etf_weighted_metrics.keys())[0]
    print(f"   {sample_etf}:")
    for key, value in list(etf_weighted_metrics[sample_etf].items())[:10]:
        if value is not None:
            print(f"     {key}: {value}")

print(f"\n{'='*80}")

## Visualize Sample Time Series

In [ ]:
import matplotlib.pyplot as plt

if etf_weighted_timeseries:
    # Plot a sample time series
    sample_etf = list(etf_weighted_timeseries.keys())[0]
    sample_ts = etf_weighted_timeseries[sample_etf]
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'{sample_etf} - Cap-Weighted Fundamental Metrics Over Time', fontsize=16)
    
    # Plot ROE
    if 'roe' in sample_ts.columns:
        sample_ts['roe'].plot(ax=axes[0, 0], title='Return on Equity (ROE)', grid=True)
        axes[0, 0].set_ylabel('ROE')
    
    # Plot Net Margin
    if 'net_margin' in sample_ts.columns:
        sample_ts['net_margin'].plot(ax=axes[0, 1], title='Net Profit Margin', grid=True)
        axes[0, 1].set_ylabel('Net Margin')
    
    # Plot Debt to Equity
    if 'debt_to_equity' in sample_ts.columns:
        sample_ts['debt_to_equity'].plot(ax=axes[1, 0], title='Debt to Equity', grid=True)
        axes[1, 0].set_ylabel('Debt/Equity')
    
    # Plot Current Ratio
    if 'current_ratio' in sample_ts.columns:
        sample_ts['current_ratio'].plot(ax=axes[1, 1], title='Current Ratio', grid=True)
        axes[1, 1].set_ylabel('Current Ratio')
    
    plt.tight_layout()
    plt.show()
else:
    print("No time series data available to plot")